# Ensemble Learning: Bias-Variance Trade-off — Decision Tree vs. Random Forest

---

## Introduction

The **bias-variance trade-off** is a fundamental concept in machine learning that describes two competing sources of prediction error:

- **Bias** — error from overly simplistic assumptions. A high-bias model underfits and misses the true structure of the data.
- **Variance** — error from excessive sensitivity to training data. A high-variance model overfits and learns noise rather than the underlying pattern.

**Decision Trees** are a classic high-variance, low-bias model. With no depth constraint, a tree memorizes its training data and produces irregular, jagged decision boundaries that do not generalize.

**Random Forest** directly addresses variance by averaging many trees, each trained on a different bootstrap sample with a random feature subset. The result is a smoother, more stable boundary with significantly better generalization.

### Why concentric circles?

The `make_circles` dataset produces two non-linearly separable ring-shaped classes with added noise. This makes decision boundary overfitting immediately visible: a single tree carves out irregular, noisy regions, while the forest recovers the smooth circular structure.

### Workflow

1. Generate and visualize the dataset
2. Fit a `DecisionTreeClassifier` and examine its decision boundary
3. Fit a `RandomForestClassifier` and compare decision boundaries
4. Report OOB score alongside train and test accuracy
5. Side-by-side comparison with accuracy annotations

---

## 1. Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

---

## 2. Dataset — Concentric Circles

We generate 500 samples arranged as two concentric rings with substantial noise (`noise=0.35`) and a tight inner ring (`factor=0.1`). The high noise level makes boundary overfitting easy to spot visually and ensures neither model achieves perfect accuracy.

In [ ]:
X, y = make_circles(n_samples=500, factor=0.1, noise=0.35, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Total samples : {X.shape[0]}')
print(f'Train         : {X_train.shape[0]}')
print(f'Test          : {X_test.shape[0]}')
print(f'Class balance : {np.bincount(y)}')

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', alpha=0.7,
            edgecolors='k', linewidths=0.3)
plt.title('make_circles — Two Concentric Classes with Noise')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.tight_layout()
plt.show()

---

## 3. Decision Boundary Helper

A reusable function that plots the decision boundary of any fitted classifier over the full feature space, with training data overlaid.

In [ ]:
def plot_boundary(clf, X, y, title, ax=None):
    """Plot the decision boundary of a fitted classifier."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))

    X_range = np.linspace(X.min() - 0.3, X.max() + 0.3, 200)
    xx1, xx2 = np.meshgrid(X_range, X_range)
    y_hat = clf.predict(np.c_[xx1.ravel(), xx2.ravel()]).reshape(xx1.shape)

    ax.contourf(xx1, xx2, y_hat, alpha=0.25, cmap='viridis')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', alpha=0.7,
               edgecolors='k', linewidths=0.3, s=25)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

---

## 4. Decision Tree — High Variance Baseline

An unpruned `DecisionTreeClassifier` partitions the feature space with axis-aligned splits until every training sample is correctly classified. On noisy data this produces a highly irregular boundary — the textbook signature of **high variance**.

The train accuracy of 1.0 paired with a lower test accuracy confirms overfitting: the tree memorizes the training labels rather than learning the circular structure.

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

dt_train_acc = accuracy_score(y_train, dt.predict(X_train))
dt_test_acc  = accuracy_score(y_test,  dt.predict(X_test))

print(f'Decision Tree  —  Train: {dt_train_acc:.4f}   Test: {dt_test_acc:.4f}')
print(f'Tree depth: {dt.get_depth()}   Leaf nodes: {dt.get_n_leaves()}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot_boundary(
    dt, X, y,
    title=f'Decision Tree  |  Train: {dt_train_acc:.3f}   Test: {dt_test_acc:.3f}',
    ax=ax
)
plt.tight_layout()
plt.show()

---

## 5. Random Forest — Variance Reduction via Averaging

A `RandomForestClassifier` with 500 trees averages predictions from trees trained on independent bootstrap samples with different feature subsets at every split. The decision boundary converges toward the true circular structure rather than memorizing individual noise points.

`oob_score=True` provides a free generalization estimate using the samples left out of each tree's bootstrap — no extra data or computation needed.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_train_acc = accuracy_score(y_train, rf.predict(X_train))
rf_test_acc  = accuracy_score(y_test,  rf.predict(X_test))
rf_oob_acc   = rf.oob_score_

print(f'Random Forest  —  Train: {rf_train_acc:.4f}   Test: {rf_test_acc:.4f}   OOB: {rf_oob_acc:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot_boundary(
    rf, X, y,
    title=f'Random Forest (500 trees)  |  Train: {rf_train_acc:.3f}   Test: {rf_test_acc:.3f}   OOB: {rf_oob_acc:.3f}',
    ax=ax
)
plt.tight_layout()
plt.show()

---

## 6. Side-by-Side Comparison

Placing both boundaries next to each other makes the variance reduction effect immediately clear. The Decision Tree produces a fragmented, noisy boundary; the Random Forest approximates the smooth circular boundary of the true data-generating process.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_boundary(
    dt, X, y,
    title=f'Decision Tree\nTrain: {dt_train_acc:.3f}   Test: {dt_test_acc:.3f}',
    ax=axes[0]
)
plot_boundary(
    rf, X, y,
    title=f'Random Forest (500 trees)\nTrain: {rf_train_acc:.3f}   Test: {rf_test_acc:.3f}   OOB: {rf_oob_acc:.3f}',
    ax=axes[1]
)

plt.suptitle('Bias-Variance Trade-off: Decision Tree vs. Random Forest', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---

## 7. Results Summary

In [ ]:
summary = pd.DataFrame({
    'Model':       ['Decision Tree', 'Random Forest (500 trees)'],
    'Train Acc':   [round(dt_train_acc, 4), round(rf_train_acc, 4)],
    'Test Acc':    [round(dt_test_acc,  4), round(rf_test_acc,  4)],
    'OOB Score':   ['-',                    round(rf_oob_acc,   4)],
    'Overfit Gap': [round(dt_train_acc - dt_test_acc, 4),
                    round(rf_train_acc - rf_test_acc, 4)]
})

print(summary.to_string(index=False))

---

## Conclusion

This notebook demonstrated the bias-variance trade-off visually using the `make_circles` dataset, where the non-linear circular boundary makes overfitting immediately visible in the decision region plots.

**Key findings:**

- The **Decision Tree** achieves a perfect train accuracy of 1.0 but drops significantly on the test set, confirming high variance. Its decision boundary is fragmented and irregular — it memorized the noise in the training data rather than learning the circular class structure.
- The **Random Forest** shows a much smaller train-test gap. By averaging 500 trees trained on different bootstrap samples with per-node feature randomization, individual tree errors cancel out and the boundary converges toward the smooth circular shape of the true data-generating process.
- The **OOB score** closely tracks the test accuracy, confirming it as a reliable free estimate of generalization performance without requiring a separate validation set.
- The **overfit gap** (train minus test accuracy) quantifies variance directly: the large gap for the Decision Tree versus the small gap for the Random Forest is a precise numerical expression of the variance reduction that ensemble averaging achieves.

**Takeaways:**

- Variance reduction through averaging is the core mechanism behind Random Forest — not any improvement in individual tree quality.
- The `make_circles` problem is a useful diagnostic benchmark: any model that produces a blocky or fragmented circular boundary is overfitting the noise.
- `n_estimators=500` is conservative; the boundary and accuracy typically stabilize well before 200 trees. The OOB convergence plot from the previous notebook is a practical guide for choosing the right cutoff.
- Controlling `max_depth` on the base trees is a direct lever for tuning the bias-variance balance: shallower trees reduce variance further but introduce some bias.